# LungView CT nodule detection — research notebook
This notebook runs the official MONAI 3D RetinaNet lung-nodule bundle, pretrained on LUNA16/LIDC-IDRI. It localizes **possible nodules** in CT volumes; it does not diagnose cancer or certify that a scan is healthy. A radiologist must review all CT imaging.

In [ ]:
!nvidia-smi
!pip -q install 'monai[fire]' nibabel itk
import json, os, pathlib, subprocess, sys
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect.'
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
BUNDLE_DIR = pathlib.Path('/content/bundles')
!python -m monai.bundle download lung_nodule_ct_detection --bundle_dir /content/bundles
BUNDLE = BUNDLE_DIR / 'lung_nodule_ct_detection'
assert (BUNDLE / 'models/model.pt').exists(), 'Model download failed.'
print('Official pretrained model ready.')

## Upload one CT volume
Upload a de-identified chest CT in `.nii` or `.nii.gz` format. Do not upload scans containing patient-identifying information. A DICOM series should first be converted to NIfTI with a clinical imaging tool such as 3D Slicer.

In [ ]:
from google.colab import files
uploaded = files.upload()
names = [name for name in uploaded if name.endswith(('.nii', '.nii.gz'))]
assert len(names) == 1, 'Upload exactly one .nii or .nii.gz CT volume.'
ct_path = pathlib.Path('/content') / names[0]
datalist = {'validation': [{'image': str(ct_path)}]}
list_path = pathlib.Path('/content/ct_input.json')
list_path.write_text(json.dumps(datalist))
print('CT ready:', ct_path.name)

In [ ]:
RESULTS = pathlib.Path('/content/results')
RESULTS.mkdir(exist_ok=True)
cmd = [sys.executable, '-m', 'monai.bundle', 'run',
       '--config_file', str(BUNDLE / 'configs/inference.json'),
       '--bundle_root', str(BUNDLE),
       '--dataset_dir', '/content',
       '--data_list_file_path', str(list_path),
       '--whether_raw_luna16', 'true',
       '--force_sliding_window', 'true',
       '--output_dir', str(RESULTS),
       '--output_filename', 'lungview_detections.json']
print('Running 3D inference. This can take several minutes…')
subprocess.run(cmd, check=True)
result_path = RESULTS / 'lungview_detections.json'
assert result_path.exists(), 'Inference finished without a result file.'
detections = json.loads(result_path.read_text())
print(json.dumps(detections, indent=2)[:12000])

In [ ]:
# Export the machine-readable detection report for LungView integration.
report = {
  'model': 'MONAI lung_nodule_ct_detection',
  'training_data': 'LUNA16 / LIDC-IDRI',
  'result': detections,
  'disclaimer': 'Research use only. Possible nodules are not a cancer diagnosis; no detection does not establish a healthy scan.'
}
export_path = pathlib.Path('/content/lungview-ct-report.json')
export_path.write_text(json.dumps(report, indent=2))
files.download(str(export_path))

## Interpretation limits
The model was developed for research nodule localization on LUNA16 CT data. Performance can change with scanner, reconstruction, population, contrast, slice spacing, and acquisition protocol. It does not determine malignancy. Do not use its output for diagnosis, screening, or treatment decisions.